In [2]:
!pip install torch torchvision pandas tqdm scikit-learn matplotlib seaborn torchmetrics


  Using cached torchmetrics-1.8.2-py3-none-any.whl.metadata (22 kB)
  Using cached lightning_utilities-0.15.2-py3-none-any.whl.metadata (5.7 kB)
Using cached torchmetrics-1.8.2-py3-none-any.whl (983 kB)
Using cached lightning_utilities-0.15.2-py3-none-any.whl (29 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchmetrics] [torchmetrics]


In [ ]:
import torch, torchvision
print("torch:", torch.__version__, torch.__file__)
print("torchvision:", torchvision.__version__, torchvision.__file__)


torch: 2.10.0+cu128 /home/na1488tr-s/.local/lib/python3.10/site-packages/torch/__init__.py
torchvision: 0.25.0+cu128 /home/na1488tr-s/.local/lib/python3.10/site-packages/torchvision/__init__.py


In [1]:
import sys
print(sys.executable)


/home/na1488tr-s/Bachelor_Project_Statistics/.venv/bin/python


In [23]:
import os 
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import torchvision.transforms as T
from torchmetrics.classification import MulticlassRecall, MulticlassAccuracy
from tqdm import tqdm
from collections import Counter

from Data_Utility.lookup_size import lookup_size_from_excel
from Data_Utility.dataset import PollenFolderWithSizeDataset

from models.basemodel import CNNWithSizeMLP

print("All libraries imported successfully!")

All libraries imported successfully!


## Preparing data

In [24]:
train_dir = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Sorted_224_sizeTrain'
test_dir = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Sorted_224_sizeTest'
test_excel_path = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Size_features/size_data.xlsx'

size_lookup, species_mean_lookup, global_mean = lookup_size_from_excel(test_excel_path)

classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {v: k for k, v in class_to_idx.items()}
#train_tf = T.Compose([T.ToTensor()]) # converts PIL → Tensor
#test_tf = T.Compose([T.ToTensor()]) # converts PIL → Tensor

train_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


train_dataset = PollenFolderWithSizeDataset(img_dir=train_dir, class_to_idx=class_to_idx, size_lookup=size_lookup, species_mean_lookup=species_mean_lookup, global_mean=global_mean, transform=train_tf)
test_dataset = PollenFolderWithSizeDataset(img_dir=test_dir, class_to_idx=class_to_idx, size_lookup=size_lookup, species_mean_lookup=species_mean_lookup, global_mean=global_mean, transform=test_tf)




Total samples: 9844
No missing size data found. All rows have valid majoraxis and minoraxis values in Excel file.


In [25]:
train_dataset.print_missing_summary()


========== Dataset Missing Size Summary =========:
Total samples: 8078
Missing size entries: 0
Filled with species mean: 0
Filled with global mean: 0

========== Per-species missing size summary: =================


In [26]:
test_dataset.print_missing_summary()


========== Dataset Missing Size Summary =========:
Total samples: 1922
Missing size entries: 0
Filled with species mean: 0
Filled with global mean: 0

========== Per-species missing size summary: =================


In [27]:
train_counts = Counter([train_dataset[i][2].item() for i in range(len(train_dataset))])




Finished processing dataset.

========== Dataset Missing Size Summary =========:
Total samples: 8078
Missing size entries: 156
Filled with species mean: 156
Filled with global mean: 0

========== Per-species missing size summary: =================

Species: {species}
   Missing: 2
   Filled with species mean: 2
   Filled with global mean: 0

Species: {species}
   Missing: 5
   Filled with species mean: 5
   Filled with global mean: 0

Species: {species}
   Missing: 1
   Filled with species mean: 1
   Filled with global mean: 0

Species: {species}
   Missing: 65
   Filled with species mean: 65
   Filled with global mean: 0

Species: {species}
   Missing: 83
   Filled with species mean: 83
   Filled with global mean: 0


In [22]:
test_counts = Counter([test_dataset[i][2].item() for i in range(len(test_dataset))])


No missing size data found in the image dataset.


In [ ]:
print("Training set class distribution:")
for k in sorted(train_counts):
    print(k, idx_to_class[int(k)], train_counts[k])

In [17]:

print("\nTest set class distribution:")
for k in sorted(test_counts):
    print(k, idx_to_class[int(k)], test_counts[k])


No missing size data found in the image dataset.

Test set class distribution:
0 Bellis perennis 142
1 Brassica napus 200
2 Capsella bursa-pastoris 183
3 Cichorium intybus 130
4 Crepis capillaris 200
5 Hieracium umbellatum 200
6 Hypochaeris radicata 200
7 Sonchus arvensis 334
8 Tragopogon pratensis 166
9 Tussilago farfara 167


In [8]:
from os import access

from sympy import Mul


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
model = CNNWithSizeMLP(num_classes=len(classes)).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Performance metric
recall_metric = MulticlassRecall(num_classes=len(classes), average = None).to(device)
accuracy_metric = MulticlassAccuracy(num_classes=len(classes), average = "micro").to(device)  




Using device: cuda


In [9]:
def train_epoch(loader):
    model.train()
    total_loss = 0
    for imgs, sizes, labels in tqdm(loader):
        imgs = imgs.to(device)
        sizes = sizes.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs, sizes)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(loader):
    model.eval()
    
    recall_metric.reset()
    accuracy_metric.reset()
    
    with torch.no_grad():
        for imgs, sizes, labels in loader:
            imgs = imgs.to(device)
            sizes = sizes.to(device)
            labels = labels.to(device)

            outputs = model(imgs, sizes)
            preds = outputs.argmax(dim=1)
            
            recall_metric.update(preds, labels)
            accuracy_metric.update(preds, labels)
            
    recall_per_class = recall_metric.compute().cpu() # tensor of shape (num_classes,)
    accuracy = accuracy_metric.compute().cpu().item() # scalar
    return recall_per_class, accuracy

## Base Model (Erik's)

In [ ]:
def split_one_sample_per_species(
    dataset,
    seed: int = 42,
    val_individuals_per_class: int = 1,
    special_val_counts: dict = None,
):
    """
    Split dataset following Erik Jurdell's experimental protocol.

    Each species contains approximately 1000 images sampled from
    3–10 individual flowers, where each individual represents a
    unique biological specimen.

    To ensure a challenging and biologically realistic evaluation,
    the split is performed at the individual level (not image level):

        • For each species, exactly one individual flower is
          reserved for validation.
        • For Bellis perennis, two individuals are used for validation
          because this species contains roughly half as many images
          per individual compared to other species.
        • All remaining individuals are used for training.

    This guarantees that validation images come from individuals
    that were not seen during training, preventing overestimation
    of model performance due to individual-specific features.

    Returns:
        train_subset (torch.utils.data.Subset)
        val_subset   (torch.utils.data.Subset)
        
    For example:
    Species A
    - Individual/flower 1: img1, img2, img3
    - Individual/flower 2: img4, img5, img6
    - Individual/flower 3: img7, img8, img9
    Each individual/flower belongs to a unique biological specimen
    
    So Erik split by individual flower, not by image for validation.
    i.e.
    For each species, exactly one individual flower was reserved for validation.
    The rest of flowers were used for training. 
    Exception: For Bellis perennis, two individuals were used for validation because this species contains roughly half as many images per individual compared to other species.
    """
    img_paths = []
    labels = []
    groups = []

    for idx, (img_path, class_name) in enumerate(dataset.samples):
        img_paths.append(img_path)
        labels.append(class_name)

        # individual ID from filename
        fn = os.path.basename(img_path)
        indiv = get_individual_id(fn)
        groups.append(indiv)

    img_paths = np.array(img_paths)
    labels = np.array(labels)
    groups = np.array(groups)

    # This ensures individuals don't mix between splits
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, val_idx = next(gss.split(img_paths, labels, groups))

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    return train_subset, val_subset

In [6]:
epochs_num = 25
best_val_acc = 0.0   # initialize before loop

for epoch in range(1, epochs_num + 1):
    train_loss = train_epoch(train_loader)
    val_recall_per_class, val_acc = eval_epoch(val_loader)
    
    print(f"Epoch {epoch}: loss {train_loss:.4f}, val_acc {val_acc:.4f}")
    print(f"Val Recall per class: {val_recall_per_class}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved new best model!")

100%|██████████| 202/202 [00:19<00:00, 10.62it/s]


Epoch 1: loss 0.3059, val_acc 0.6885
Val Recall per class: tensor([0.8508, 0.9883, 0.8431, 1.0000, 0.4052, 0.4625, 0.0652, 0.1357, 0.8690,
        0.9942])
Saved new best model!


100%|██████████| 202/202 [00:18<00:00, 10.99it/s]


Epoch 2: loss 0.1254, val_acc 0.7988
Val Recall per class: tensor([1.0000, 0.9942, 0.9804, 0.2222, 0.9477, 0.5875, 0.9058, 0.3357, 0.9940,
        1.0000])
Saved new best model!


100%|██████████| 202/202 [00:18<00:00, 11.04it/s]


...

Epoch 23: loss 0.0132, val_acc 0.8793
Val Recall per class: tensor([0.7127, 0.9825, 0.8758, 0.9833, 0.9869, 1.0000, 0.6594, 0.8500, 0.7143,
        1.0000])


100%|██████████| 202/202 [00:18<00:00, 10.94it/s]


Epoch 24: loss 0.0353, val_acc 0.9808
Val Recall per class: tensor([0.9890, 0.9649, 1.0000, 0.9833, 0.9477, 0.9563, 0.9928, 0.9786, 1.0000,
        0.9942])


100%|██████████| 202/202 [00:18<00:00, 10.92it/s]


Epoch 25: loss 0.0187, val_acc 0.9406
Val Recall per class: tensor([0.9945, 0.9883, 0.9869, 0.5667, 0.9804, 0.9563, 1.0000, 0.9857, 0.9940,
        1.0000])


In [11]:
print("\nTraining complete.")
print("Loading best model for final test evaluation...")

model.load_state_dict(torch.load("best_model.pth"))

test_recall_per_class, test_acc = eval_epoch(test_loader)

print(f"\nFinal Test Accuracy: {test_acc:.4f}")
print(f"Final Test Recall per class: {test_recall_per_class}")


Training complete.
Loading best model for final test evaluation...

Final Test Accuracy: 0.7513
Final Test Recall per class: tensor([0.2042, 1.0000, 1.0000, 1.0000, 0.0050, 0.9300, 0.8300, 0.6647, 0.9639,
        1.0000])


In [7]:
# Splitting the training dataset into training and validation sets
import random


val_ratio = 0.2
n_total = len(train_dataset)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

# for reproducibility
g = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(train_dataset, [n_train, n_val], generator=g)

print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True, num_workers=0)

Training samples: 5171, Validation samples: 1292, Test samples: 1922
